In [3]:
from dotenv import load_dotenv
import os
import requests
import duckdb as dd
load_dotenv()
conn = dd.connect('spotify_all_years.db')
TICKETMASTER_API_KEY = os.getenv('consumer_key')

In [25]:
def get_event_details(query):
    base_url = "https://app.ticketmaster.com/discovery/v2/events.json"
    params = {
        "apikey": TICKETMASTER_API_KEY,
        "keyword": query,
        "classificationName": "music",  
        "geoPoint": "dpsby4",   #geohash for detroit            
        "radius": 150,                   
        "unit": "miles",
        "size": 5,
        "countryCode": "US",
        "source": "ticketmaster"                      
    }
    response = requests.get(base_url, params=params)
    list_of_valid_artists = []
    if response.status_code == 200 and "_embedded" in response.json():
        for event in response.json()["_embedded"]["events"]:
            event_dict = {
                "artist_name": event['name'],
                "ticket_link": event['url'],
                "artist_image": max(event["images"], key=lambda x: x["width"])["url"],
                "concert_date": event['dates']['start']['localDate'] ,
                "venue":event['_embedded']['venues'][0]['name'] ,
                "city": event['_embedded']['venues'][0]['city']['name'],
                "address":event['_embedded']['venues'][0]['address']['line1'] ,
            }
            list_of_valid_artists.append(event_dict)
    elif response.status_code != 200:
        print(f"API error for {query}: {response.status_code}")
    else: 
        return []
    
    return list_of_valid_artists

In [26]:
import time
top_50 = conn.execute("SELECT * FROM superfan_scores").df().head(50)
list_of_artists = top_50["artist_name"].tolist()
touring_artists = []
for i in list_of_artists:
    touring_artists += get_event_details(i)
    time.sleep(0.2)

In [ ]:
from jinja2 import Environment, FileSystemLoader
from datetime import date
env = Environment(loader=FileSystemLoader("."))
template = env.get_template('email_template.html')
touring_artists = sorted(touring_artists, key=lambda x: x["concert_date"])
html_output = template.render(events = touring_artists, date = date.today().strftime("%B %d, %Y"))

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

load_dotenv()
app_password = os.getenv('app_password')
sender = os.getenv('gmail_account')
receiver = os.getenv('gmail_account')

msg = MIMEMultipart("alternative")
msg["Subject"] = "Your Weekly Concert Digest"
msg["From"] = sender
msg["To"] = receiver
msg.attach(MIMEText(html_output, "html"))

with smtplib.SMTP("smtp.gmail.com", 587) as server:
    server.starttls()
    server.login(sender, app_password)
    server.sendmail(sender, receiver, msg.as_string())